# 3. Candidate Table Generation

The single-user engine can enumerate a large active table for one deployment distance, but the scheduler does not need every row. This notebook shows how that full-frame active table is projected onto the stored scheduler-facing contract and then strictly pruned into a precomputed frontier for each distance bin.


## 1. Build the precomputed table across the distance grid

The stored artifact is keyed by distance bins. Each bin keeps only full-frame rows and only the non-dominated alternatives within each PA family.


In [ ]:
import sys
from pathlib import Path
from IPython.display import display

repo_root = Path.cwd().resolve()
if not (repo_root / "src").exists():
    repo_root = repo_root.parent

for path in (repo_root / "notebooks", repo_root / "src"):
    resolved = str(path.resolve())
    if resolved not in sys.path:
        sys.path.insert(0, resolved)

from helpers.candidate_table_generation_helpers import (
    build_candidate_table_generation_artifacts,
    plot_frontier_compaction,
    plot_pruned_frontier,
)

artifacts = build_candidate_table_generation_artifacts(distance_m=200)


In [ ]:
display(artifacts.distance_summary.head(10))


## 2. One distance slice before and after pruning

For one selected distance bin, the notebook rebuilds the full-frame active rows before pruning. This makes the table-generation step explicit: first keep only the scheduler-facing row contract, then remove rows that are strictly dominated on the same PA family.


In [ ]:
display(artifacts.pruning_summary)
display(artifacts.full_frame_candidate_table.head(12))
display(artifacts.pruned_frontier_table.head(12))


In [ ]:
plot_frontier_compaction(artifacts.pruning_summary)


## 3. The stored frontier for one distance bin

The retained rows are the rows later notebooks will assign to users. They already carry the active-rate and active-power values needed by the TDMA layer, but they are much smaller than the raw full-frame table.


In [ ]:
plot_pruned_frontier(
    artifacts.pruned_frontier_table,
    pa_label_map=artifacts.pa_label_map,
)


The next notebook generates users over the day. After that, the lookup notebook shows how one active user is mapped onto this stored distance-binned table before the joint TDMA scheduler is allowed to choose one row per user.
